# Animal Detector Inference

In [ ]:
# Install dependencies (run once)
%pip -q install ultralytics opencv-python

# Imports
from pathlib import Path
from ultralytics import YOLO
import cv2

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Configuration
input_type = 'video'  # Options: 'video', 'frames', 'folder'
input_path = 'path/to/video.mp4'  # or folder path
output_fps = 30
output_video_path = 'output_labeled_video.mp4'
target_class_index = 0
confidence_threshold = 0.25
pixels_per_unit_distance = None
runs_detect_dir = Path('runs/detect')
starting_model = 'yolov8n.pt'

In [ ]:
def find_most_recent_weights(search_dir: Path) -> Path | None:
    if not search_dir.exists():
        return None

    weights_files = list(search_dir.rglob("best.pt"))
    if not weights_files:
        return None

    weights_files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return weights_files[0]


# Load model with fallback and validation
recent_weights = find_most_recent_weights(runs_detect_dir)
inference_weights_path = Path(recent_weights) if recent_weights is not None else Path(starting_model)

if not inference_weights_path.exists():
    raise FileNotFoundError(
        f"No model weights found. Expected trained weights in {runs_detect_dir} or fallback file {starting_model}."
    )

try:
    inference_model = YOLO(str(inference_weights_path))
except Exception as exc:
    raise RuntimeError(f"Failed to load YOLO model from {inference_weights_path}: {exc}") from exc

print(f"Loaded model: {inference_weights_path}")

In [ ]:
# Input handling
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv'}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
VALID_INPUT_TYPES = {'video', 'frames', 'folder'}


def get_input_manifest(input_path: str, input_type: str) -> dict:
    normalized_type = str(input_type).strip().lower()
    if normalized_type not in VALID_INPUT_TYPES:
        raise ValueError(f"Invalid input_type '{input_type}'. Choose from {sorted(VALID_INPUT_TYPES)}.")

    path = Path(input_path)
    if not path.exists():
        raise FileNotFoundError(f"Input path does not exist: {path}")

    if normalized_type == 'video':
        if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
            raise ValueError(f"For input_type='video', provide a valid video file: {sorted(VIDEO_EXTENSIONS)}")
        return {'input_type': normalized_type, 'videos': [path], 'images': []}

    if not path.is_dir():
        raise ValueError(f"For input_type='{normalized_type}', provide a directory path.")

    entries = [p for p in sorted(path.iterdir()) if p.is_file()]
    images = [p for p in entries if p.suffix.lower() in IMAGE_EXTENSIONS]
    videos = [p for p in entries if p.suffix.lower() in VIDEO_EXTENSIONS]

    if normalized_type == 'frames':
        if videos:
            raise ValueError("input_type='frames' only accepts image files, but videos were found.")
        if not images:
            raise RuntimeError(f"No frame images found in {path}. Supported: {sorted(IMAGE_EXTENSIONS)}")
        return {'input_type': normalized_type, 'videos': [], 'images': images}

    if not images and not videos:
        raise RuntimeError(
            f"No supported files found in {path}. Supported images={sorted(IMAGE_EXTENSIONS)}, videos={sorted(VIDEO_EXTENSIONS)}"
        )
    return {'input_type': normalized_type, 'videos': videos, 'images': images}


input_manifest = get_input_manifest(input_path, input_type)
print(
    f"Resolved input: type={input_manifest['input_type']}, "
    f"images={len(input_manifest['images'])}, videos={len(input_manifest['videos'])}"
)

In [ ]:
def detect_best_center_in_frame(frame_path: Path):
    prediction_results = inference_model.predict(
        source=str(frame_path),
        imgsz=640,
        conf=confidence_threshold,
        verbose=False,
    )
    best_match = None
    for result in prediction_results:
        if result.boxes is None or len(result.boxes) == 0:
            continue
        boxes = result.boxes
        class_ids = boxes.cls.cpu().numpy().astype(int)
        confidences = boxes.conf.cpu().numpy()
        bounding_boxes = boxes.xyxy.cpu().numpy()
        for i in range(len(class_ids)):
            if class_ids[i] != target_class_index:
                continue
            if best_match is None or confidences[i] > best_match["confidence"]:
                x1, y1, x2, y2 = bounding_boxes[i]
                center_x = (x1 + x2) / 2.0
                center_y = (y1 + y2) / 2.0
                best_match = {
                    "center_x": float(center_x),
                    "center_y": float(center_y),
                    "confidence": float(confidences[i]),
                }
    return best_match

In [ ]:
def extract_best_match_from_result(result, target_class_idx: int):
    if result.boxes is None or len(result.boxes) == 0:
        return None

    boxes = result.boxes
    class_ids = boxes.cls.cpu().numpy().astype(int)
    confidences = boxes.conf.cpu().numpy()
    bounding_boxes = boxes.xyxy.cpu().numpy()

    best_match = None
    for i in range(len(class_ids)):
        if class_ids[i] != target_class_idx:
            continue
        if best_match is None or confidences[i] > best_match["confidence"]:
            x1, y1, x2, y2 = bounding_boxes[i]
            best_match = {
                "center_x": float((x1 + x2) / 2.0),
                "center_y": float((y1 + y2) / 2.0),
                "confidence": float(confidences[i]),
            }
    return best_match


def adjust_video_fps(input_video_path, output_video_path, target_fps: int):
    input_video_path = Path(input_video_path)
    output_video_path = Path(output_video_path)
    output_video_path.parent.mkdir(parents=True, exist_ok=True)

    if target_fps <= 0:
        raise ValueError(f"target_fps must be > 0, got {target_fps}")

    cap = cv2.VideoCapture(str(input_video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Error opening video: {input_video_path}")

    input_fps = cap.get(cv2.CAP_PROP_FPS)
    if not input_fps or input_fps <= 0:
        input_fps = float(target_fps)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if width <= 0 or height <= 0:
        cap.release()
        raise RuntimeError(f"Invalid video dimensions in: {input_video_path}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_video_path), fourcc, float(target_fps), (width, height))
    if not out.isOpened():
        cap.release()
        raise RuntimeError(f"Could not open video writer for: {output_video_path}")

    # Sampling strategy to preserve approximate runtime at target_fps
    frame_index = 0
    written = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        should_write = int(frame_index * target_fps / input_fps) > int((frame_index - 1) * target_fps / input_fps) if frame_index > 0 else True
        if should_write:
            out.write(frame)
            written += 1
        frame_index += 1

    cap.release()
    out.release()

    if written == 0:
        raise RuntimeError(f"No frames were written while adjusting FPS for: {input_video_path}")

    print(f"Adjusted video saved to {output_video_path} at {target_fps} FPS ({written} frames)")


def create_video_from_images(image_paths, output_video_path, fps: int):
    output_video_path = Path(output_video_path)
    output_video_path.parent.mkdir(parents=True, exist_ok=True)

    if fps <= 0:
        raise ValueError(f"fps must be > 0, got {fps}")
    if not image_paths:
        raise RuntimeError("No images provided to create_video_from_images.")

    first_frame = cv2.imread(str(image_paths[0]))
    if first_frame is None:
        raise RuntimeError(f"Unable to read first image: {image_paths[0]}")

    height, width = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_video_path), fourcc, float(fps), (width, height))
    if not out.isOpened():
        raise RuntimeError(f"Could not open video writer for: {output_video_path}")

    written = 0
    for img_path in image_paths:
        frame = cv2.imread(str(img_path))
        if frame is None:
            print(f"Skipping unreadable image: {img_path}")
            continue

        if frame.shape[1] != width or frame.shape[0] != height:
            frame = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)

        out.write(frame)
        written += 1

    out.release()

    if written == 0:
        raise RuntimeError("No valid images were written to output video.")

    print(f"Video created from images: {output_video_path} at {fps} FPS ({written} frames)")

In [ ]:
# Processing
import time


def build_rows_for_results(results_obj, source_name: str):
    rows = []
    for frame_idx, result in enumerate(results_obj):
        best_match = extract_best_match_from_result(result, target_class_index)
        rows.append({
            "source": source_name,
            "frame": f"frame_{frame_idx}",
            "frame_index": frame_idx,
            "center_x": best_match["center_x"] if best_match else None,
            "center_y": best_match["center_y"] if best_match else None,
            "confidence": best_match["confidence"] if best_match else None,
            "distance_from_prev": None,
            "cumulative_distance": 0.0,
        })
    return rows


def run_predict(source, run_name: str):
    return inference_model.predict(
        source=source,
        imgsz=640,
        conf=confidence_threshold,
        save=True,
        save_txt=False,
        save_conf=True,
        verbose=False,
        project=str(runs_detect_dir),
        name=run_name,
        exist_ok=False,
    )


table_rows = []
produced_videos = []
base_output_path = Path(output_video_path)
base_output_path.parent.mkdir(parents=True, exist_ok=True)

if confidence_threshold < 0 or confidence_threshold > 1:
    raise ValueError(f"confidence_threshold must be in [0, 1], got {confidence_threshold}")

timestamp = int(time.time())

if input_manifest['input_type'] == 'video':
    source_video = input_manifest['videos'][0]
    run_name = f"predict_video_{timestamp}"
    results = run_predict(str(source_video), run_name)
    table_rows.extend(build_rows_for_results(results, source_video.name))

    run_dir = runs_detect_dir / run_name
    saved_video = run_dir / source_video.name
    if not saved_video.exists():
        found = list(run_dir.rglob("*.mp4"))
        if not found:
            raise RuntimeError(f"No labeled video output found in {run_dir}")
        saved_video = max(found, key=lambda p: p.stat().st_mtime)

    adjust_video_fps(saved_video, base_output_path, output_fps)
    produced_videos.append(base_output_path)

else:
    # Images batch (frames or mixed folder)
    images = input_manifest['images']
    if images:
        run_name = f"predict_images_{timestamp}"
        results = run_predict([str(p) for p in images], run_name)
        table_rows.extend(build_rows_for_results(results, "images_batch"))

        run_dir = runs_detect_dir / run_name
        labeled_images = sorted(
            [p for p in run_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        ) if run_dir.exists() else []

        if not labeled_images:
            raise RuntimeError(f"No labeled images found in {run_dir}")

        image_video_output = base_output_path if input_manifest['input_type'] == 'frames' else base_output_path.with_name(f"{base_output_path.stem}_images.mp4")
        create_video_from_images(labeled_images, image_video_output, output_fps)
        produced_videos.append(image_video_output)

    # Each video in mixed folder
    for idx, video_path in enumerate(input_manifest['videos']):
        run_name = f"predict_folder_video_{timestamp}_{idx}"
        results = run_predict(str(video_path), run_name)
        table_rows.extend(build_rows_for_results(results, video_path.name))

        run_dir = runs_detect_dir / run_name
        saved_video = run_dir / video_path.name
        if not saved_video.exists():
            found = list(run_dir.rglob("*.mp4"))
            if not found:
                raise RuntimeError(f"No labeled video output found in {run_dir}")
            saved_video = max(found, key=lambda p: p.stat().st_mtime)

        folder_video_output = base_output_path.with_name(f"{video_path.stem}_labeled.mp4")
        adjust_video_fps(saved_video, folder_video_output, output_fps)
        produced_videos.append(folder_video_output)

if not table_rows:
    raise RuntimeError("No detections were processed; table_rows is empty.")

print("Generated videos:")
for video in produced_videos:
    print(f"- {video}")

In [ ]:
# Compute distances, display, and save
if not table_rows:
    raise RuntimeError("No rows available to summarize. Run the processing cell first.")


def apply_distance_metrics(rows, pixels_per_unit=None):
    previous_by_source = {}
    cumulative_by_source = {}

    for row in rows:
        source = row["source"]
        if source not in cumulative_by_source:
            cumulative_by_source[source] = 0.0
            previous_by_source[source] = None

        if row["center_x"] is None or row["center_y"] is None:
            row["cumulative_distance"] = cumulative_by_source[source]
            continue

        previous_center = previous_by_source[source]
        if previous_center is not None:
            delta_x = row["center_x"] - previous_center["center_x"]
            delta_y = row["center_y"] - previous_center["center_y"]
            distance = float((delta_x ** 2 + delta_y ** 2) ** 0.5)
            if pixels_per_unit:
                if float(pixels_per_unit) <= 0:
                    raise ValueError("pixels_per_unit_distance must be > 0 when provided.")
                distance = distance / float(pixels_per_unit)
            cumulative_by_source[source] += distance
            row["distance_from_prev"] = distance

        row["cumulative_distance"] = cumulative_by_source[source]
        previous_by_source[source] = {
            "center_x": row["center_x"],
            "center_y": row["center_y"],
        }


apply_distance_metrics(table_rows, pixels_per_unit_distance)

output_csv_path = Path(output_video_path).with_suffix('.csv')
output_csv_path.parent.mkdir(parents=True, exist_ok=True)

import csv
with open(output_csv_path, "w", newline="", encoding="utf-8") as file_handle:
    writer = csv.DictWriter(file_handle, fieldnames=table_rows[0].keys())
    writer.writeheader()
    writer.writerows(table_rows)

print(f"Saved CSV to {output_csv_path}")
print(f"Total rows: {len(table_rows)}")
print("Preview:")
for row in table_rows[:5]:
    print(row)